# Part 2 — RFM Segmentation & Retention Strategy
**D2C Customer Churn Intelligence Capstone**

**Snapshot date:** `2025-09-30` | **Customers:** 2,400 | **Target:** `churn_next_60d`

---
### Notebook Structure
1. Setup & Data Loading
2. RFM Feature Engineering
3. RFM Scoring & Segmentation
4. Enrichment with Non-RFM Signals
5. Segment Profiles & Visualisation
6. Retention Strategy & Budget Prioritisation
7. Manual Review Cases (10 customers)
8. Export segments.csv

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (10, 4),
                     'axes.titlesize': 13, 'axes.labelsize': 11})

SNAPSHOT = pd.Timestamp('2025-09-30')
print('Libraries loaded ✓')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# !! UPDATE this path to match your Drive folder
DATA_DIR = '/content/drive/MyDrive/d2c_churn_data/'

customers     = pd.read_csv(DATA_DIR + 'customers.csv', parse_dates=['signup_date'])
orders        = pd.read_csv(DATA_DIR + 'orders.csv', parse_dates=['order_date'])
tickets       = pd.read_csv(DATA_DIR + 'support_tickets.csv', parse_dates=['ticket_date'])
web           = pd.read_csv(DATA_DIR + 'web_events_snapshot.csv')
labels        = pd.read_csv(DATA_DIR + 'churn_labels.csv')
interventions = pd.read_csv(DATA_DIR + 'intervention_history.csv')

# Clean orders: remove _DUP and post-snapshot
orders = orders[~orders['order_id'].str.endswith('_DUP')].copy()
orders_pre = orders[orders['order_date'] <= SNAPSHOT].copy()

print(f'Pre-snapshot orders: {len(orders_pre):,}')
print(f'Customers: {len(customers):,}')

## 2. RFM Feature Engineering

In [ ]:
# ── Build RFM from pre-snapshot orders ────────────────────────────────────────
rfm = orders_pre.groupby('customer_id').agg(
    last_order_date = ('order_date', 'max'),
    frequency       = ('order_id', 'count'),
    monetary        = ('gross_amount', 'sum'),
).reset_index()

rfm['recency_days'] = (SNAPSHOT - rfm['last_order_date']).dt.days

# Left-join to customers to include zero-order customers
rfm = customers[['customer_id']].merge(rfm, on='customer_id', how='left')

# Zero-order customers: fill with worst recency, 0 frequency, 0 monetary
max_recency = rfm['recency_days'].max()
rfm['recency_days'] = rfm['recency_days'].fillna(max_recency + 30)  # never ordered = worst recency
rfm['frequency']    = rfm['frequency'].fillna(0).astype(int)
rfm['monetary']     = rfm['monetary'].fillna(0.0)

print(f'RFM table shape: {rfm.shape}')
display(rfm.describe().round(1))

In [ ]:
# ── RFM Score (1-5 quintiles) ─────────────────────────────────────────────────
# Recency: lower days = better = higher score
rfm['R_score'] = pd.qcut(rfm['recency_days'], q=5, labels=[5,4,3,2,1], duplicates='drop').astype(int)

# Frequency: higher = better (handle ties with rank-based binning)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5], duplicates='drop').astype(int)

# Monetary: higher = better
rfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'), q=5, labels=[1,2,3,4,5], duplicates='drop').astype(int)

rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

print('RFM Score distribution:')
print(rfm['RFM_score'].describe().round(1))
print(f'\nR/F/M score sample:')
display(rfm[['customer_id','recency_days','frequency','monetary','R_score','F_score','M_score','RFM_score']].head(10))

## 3. RFM Segmentation (5+ Segments)

In [ ]:
# ── Segment assignment logic ──────────────────────────────────────────────────
# Using R, F, M scores to define business-meaningful segments

def assign_segment(row):
    R, F, M = row['R_score'], row['F_score'], row['M_score']
    total = row['RFM_score']
    
    # Champions: recent, frequent, high-spend
    if R >= 4 and F >= 4 and M >= 4:
        return 'Champions'
    
    # Loyal Customers: frequent buyers, may not be highest spend
    if F >= 4 and R >= 3:
        return 'Loyal Customers'
    
    # New Customers: very recent but low frequency
    if R >= 4 and F <= 2:
        return 'New Customers'
    
    # At-Risk: used to be good (F/M decent) but recency is poor
    if R <= 2 and (F >= 3 or M >= 3):
        return 'At-Risk'
    
    # Dormant: low on everything
    if R <= 2 and F <= 2 and M <= 2:
        return 'Dormant'
    
    # Promising: moderate across the board
    return 'Promising'

rfm['segment_name'] = rfm.apply(assign_segment, axis=1)

seg_counts = rfm['segment_name'].value_counts()
print('Segment Distribution:')
for seg, cnt in seg_counts.items():
    print(f'  {seg:<20} {cnt:>5} customers ({cnt/len(rfm)*100:.1f}%)')

In [ ]:
# ── Validate segments vs churn ────────────────────────────────────────────────
rfm_churn = rfm.merge(labels[['customer_id','churn_next_60d']], on='customer_id')

seg_profile = rfm_churn.groupby('segment_name').agg(
    n_customers    = ('customer_id', 'count'),
    avg_recency    = ('recency_days', 'mean'),
    avg_frequency  = ('frequency', 'mean'),
    avg_monetary   = ('monetary', 'mean'),
    churn_rate     = ('churn_next_60d', 'mean'),
).round(2)
seg_profile['churn_rate_pct'] = (seg_profile['churn_rate'] * 100).round(1)
seg_profile = seg_profile.sort_values('churn_rate', ascending=False)

print('Segment Profiles (sorted by churn rate):')
display(seg_profile)

In [ ]:
# ── Chart 1: Segment churn rates ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

overall_churn = labels['churn_next_60d'].mean()

# Bar chart: churn rate by segment
seg_sorted = seg_profile.sort_values('churn_rate', ascending=True)
colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(seg_sorted)))
axes[0].barh(seg_sorted.index, seg_sorted['churn_rate_pct'], color=colors, edgecolor='white')
axes[0].axvline(overall_churn*100, color='black', linestyle='--', linewidth=1, label=f'Overall ({overall_churn*100:.1f}%)')
axes[0].set_xlabel('Churn rate (%)')
axes[0].set_title('Churn Rate by Segment')
axes[0].legend()
for i, (idx, row) in enumerate(seg_sorted.iterrows()):
    axes[0].text(row['churn_rate_pct']+0.5, i, f"{row['churn_rate_pct']:.1f}%", va='center', fontsize=9)

# Pie chart: segment distribution
seg_pie = rfm['segment_name'].value_counts()
axes[1].pie(seg_pie.values, labels=seg_pie.index, autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('Set2', len(seg_pie)))
axes[1].set_title('Segment Size Distribution')

plt.suptitle('Chart 1 — RFM Segment Overview', fontweight='bold')
plt.tight_layout()
plt.savefig('chart_p2_01_segment_overview.png', bbox_inches='tight')
plt.show()

## 4. Enrichment with Non-RFM Signals

Adding **support ticket** and **web activity** signals to make segments more actionable.

In [ ]:
# ── Non-RFM Signal 1: Support ticket features ────────────────────────────────
ticket_feat = tickets.groupby('customer_id').agg(
    ticket_count       = ('ticket_id', 'count'),
    avg_sentiment      = ('sentiment_score', 'mean'),
    pct_reopened       = ('reopened', 'mean'),
    negative_tickets   = ('sentiment_score', lambda x: (x < 0).sum()),
).reset_index()

# ── Non-RFM Signal 2: Web/app activity ────────────────────────────────────────
web_feat = web[['customer_id', 'sessions_30d', 'product_views_30d',
                'cart_adds_30d', 'email_opens_30d', 'last_visit_days_ago']].copy()

# ── Non-RFM Signal 3: Return rate ────────────────────────────────────────────
return_feat = orders_pre.groupby('customer_id').agg(
    return_rate   = ('returned', 'mean'),
    avg_discount  = ('discount_pct', 'mean'),
).reset_index()

# ── Non-RFM Signal 4: Campaign history ───────────────────────────────────────
campaign_feat = interventions[['customer_id', 'last_campaign_received',
                               'manual_priority_bucket']].copy()

# ── Merge everything ─────────────────────────────────────────────────────────
enriched = rfm.merge(ticket_feat, on='customer_id', how='left')
enriched = enriched.merge(web_feat, on='customer_id', how='left')
enriched = enriched.merge(return_feat, on='customer_id', how='left')
enriched = enriched.merge(campaign_feat, on='customer_id', how='left')
enriched = enriched.merge(labels[['customer_id','churn_next_60d']], on='customer_id', how='left')

# Fill NaNs for customers with no tickets/orders
enriched['ticket_count']     = enriched['ticket_count'].fillna(0).astype(int)
enriched['avg_sentiment']    = enriched['avg_sentiment'].fillna(0.0)
enriched['negative_tickets'] = enriched['negative_tickets'].fillna(0).astype(int)
enriched['pct_reopened']     = enriched['pct_reopened'].fillna(0.0)
enriched['return_rate']      = enriched['return_rate'].fillna(0.0)
enriched['avg_discount']     = enriched['avg_discount'].fillna(0.0)

print(f'Enriched table: {enriched.shape[0]:,} customers × {enriched.shape[1]} columns')
print(f'\nNon-RFM signals added: ticket_count, avg_sentiment, negative_tickets, pct_reopened,')
print(f'  sessions_30d, product_views_30d, cart_adds_30d, email_opens_30d,')
print(f'  last_visit_days_ago, return_rate, avg_discount, campaign history')

In [ ]:
# ── Refine segments using non-RFM signals ────────────────────────────────────
# Add a 'Discount-Dependent' sub-segment for high-discount buyers
# Add a 'High-Value Unhappy' segment for high-M but negative support

def refine_segment(row):
    seg = row['segment_name']
    
    # High-Value Unhappy: good spend but has negative support experience
    if row['M_score'] >= 4 and row['negative_tickets'] >= 2 and seg != 'Champions':
        return 'High-Value Unhappy'
    
    # Discount-Dependent: >35% avg discount usage
    if row['avg_discount'] > 0.35 and seg in ['Promising', 'At-Risk']:
        return 'Discount-Dependent'
    
    return seg

enriched['segment_name'] = enriched.apply(refine_segment, axis=1)

seg_counts_v2 = enriched['segment_name'].value_counts()
print('Refined Segment Distribution:')
for seg, cnt in seg_counts_v2.items():
    print(f'  {seg:<25} {cnt:>5} ({cnt/len(enriched)*100:.1f}%)')

## 5. Segment Profiles & Visualisation

In [ ]:
# ── Full segment profile table ────────────────────────────────────────────────
profile = enriched.groupby('segment_name').agg(
    n_customers     = ('customer_id', 'count'),
    avg_recency     = ('recency_days', 'mean'),
    avg_frequency   = ('frequency', 'mean'),
    avg_monetary    = ('monetary', 'mean'),
    avg_ticket_cnt  = ('ticket_count', 'mean'),
    avg_sentiment   = ('avg_sentiment', 'mean'),
    avg_sessions    = ('sessions_30d', 'mean'),
    avg_return_rate = ('return_rate', 'mean'),
    avg_discount    = ('avg_discount', 'mean'),
    churn_rate      = ('churn_next_60d', 'mean'),
).round(2)
profile['churn_rate_pct'] = (profile['churn_rate'] * 100).round(1)
profile = profile.sort_values('churn_rate', ascending=False)

print('Complete Segment Profiles:')
display(profile)

In [ ]:
# ── Chart 2: Segment heatmap ──────────────────────────────────────────────────
heat_cols = ['avg_recency','avg_frequency','avg_monetary','avg_ticket_cnt',
             'avg_sessions','avg_return_rate','avg_discount','churn_rate_pct']

# Normalize for heatmap
heat_data = profile[heat_cols].copy()
heat_norm = (heat_data - heat_data.min()) / (heat_data.max() - heat_data.min())

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(heat_norm, annot=heat_data.values, fmt='.1f', cmap='RdYlGn_r',
            xticklabels=heat_cols, yticklabels=profile.index,
            linewidths=0.5, ax=ax)
ax.set_title('Chart 2 — Segment Profile Heatmap (values shown, colour = relative scale)')
plt.tight_layout()
plt.savefig('chart_p2_02_segment_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 3: Scatter — Recency vs Monetary coloured by segment ────────────────
fig, ax = plt.subplots(figsize=(12, 6))

segments_list = enriched['segment_name'].unique()
palette = dict(zip(segments_list, sns.color_palette('tab10', len(segments_list))))

for seg in segments_list:
    subset = enriched[enriched['segment_name'] == seg]
    ax.scatter(subset['recency_days'], subset['monetary'].clip(upper=20000),
               label=seg, alpha=0.5, s=20, color=palette[seg])

ax.set_xlabel('Recency (days since last order)')
ax.set_ylabel('Monetary (total spend INR, capped at 20k)')
ax.set_title('Chart 3 — Recency vs Monetary by Segment')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('chart_p2_03_recency_vs_monetary.png', bbox_inches='tight')
plt.show()

## 6. Retention Strategy & Budget Prioritisation

In [ ]:
# ── Budget allocation logic ───────────────────────────────────────────────────
TOTAL_BUDGET = 50000  # INR — limited campaign budget

strategy = pd.DataFrame({
    'Segment': ['Dormant', 'At-Risk', 'High-Value Unhappy', 'Discount-Dependent',
                'Promising', 'New Customers', 'Loyal Customers', 'Champions'],
    'Priority': [6, 1, 2, 5, 3, 4, 7, 8],
    'Retention Action': [
        'Minimal: send a survey asking why they left; do not invest heavily',
        'URGENT: personalised win-back email + exclusive limited-time offer (free shipping / small discount)',
        'HIGH: route to dedicated support; resolve open tickets FIRST, then offer loyalty upgrade',
        'Shift value: offer loyalty points / cashback instead of discounts; break discount dependency',
        'Nurture: product recommendations + educational content; light nudge with bundle offer',
        'Onboard: welcome series with product tips, first-reorder incentive (10% off next)',
        'Reward: early access to new launches + referral programme; no discount needed',
        'Celebrate: VIP perks, handwritten thank-you, ambassador programme; zero discount',
    ],
    'Budget Share %': [2, 30, 25, 8, 15, 12, 5, 3],
})
strategy['Budget INR'] = (strategy['Budget Share %'] / 100 * TOTAL_BUDGET).astype(int)
strategy = strategy.sort_values('Priority')

print(f'Total campaign budget: ₹{TOTAL_BUDGET:,}')
print()
display(strategy[['Priority','Segment','Retention Action','Budget Share %','Budget INR']])

In [ ]:
# ── Chart 4: Budget allocation ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

strat_sorted = strategy.sort_values('Budget Share %', ascending=True)
axes[0].barh(strat_sorted['Segment'], strat_sorted['Budget INR'],
             color=sns.color_palette('viridis', len(strat_sorted)), edgecolor='white')
axes[0].set_xlabel('Budget (INR)')
axes[0].set_title('Budget Allocation by Segment')
for i, (_, row) in enumerate(strat_sorted.iterrows()):
    axes[0].text(row['Budget INR']+200, i, f"₹{row['Budget INR']:,}", va='center', fontsize=9)

# Churn rate vs budget priority
strat_churn = strategy.merge(
    enriched.groupby('segment_name')['churn_next_60d'].mean().reset_index(),
    left_on='Segment', right_on='segment_name', how='left'
)
axes[1].scatter(strat_churn['Priority'], strat_churn['churn_next_60d']*100,
                s=strat_churn['Budget INR']/5, alpha=0.7, c='coral', edgecolors='darkred')
for _, row in strat_churn.iterrows():
    axes[1].annotate(row['Segment'], (row['Priority'], row['churn_next_60d']*100+1),
                     fontsize=7, ha='center')
axes[1].set_xlabel('Priority (1=highest)')
axes[1].set_ylabel('Churn rate (%)')
axes[1].set_title('Priority vs Churn Rate (bubble size = budget)')

plt.suptitle('Chart 4 — Retention Budget Strategy', fontweight='bold')
plt.tight_layout()
plt.savefig('chart_p2_04_budget_strategy.png', bbox_inches='tight')
plt.show()

## 7. Manual Review Cases (10 Customers)

> These are customers where the retention decision is **not obvious** from segment membership alone.

In [ ]:
# ── Select 10 ambiguous cases ─────────────────────────────────────────────────
# Look for customers at segment boundaries or with conflicting signals

# Case type 1: High-spending At-Risk with recent web visits (mixed signals)
case1 = enriched[(enriched['segment_name']=='At-Risk') & 
                  (enriched['monetary'] > enriched['monetary'].quantile(0.75)) &
                  (enriched['sessions_30d'] > 3)].head(3)

# Case type 2: Champions with negative tickets (trouble brewing)
case2 = enriched[(enriched['segment_name']=='Champions') & 
                  (enriched['negative_tickets'] >= 1)].head(2)

# Case type 3: New customers with very high first spend (high potential or one-off?)
case3 = enriched[(enriched['segment_name']=='New Customers') & 
                  (enriched['monetary'] > enriched['monetary'].quantile(0.80))].head(2)

# Case type 4: Dormant with previous high campaign engagement
case4 = enriched[(enriched['segment_name']=='Dormant') & 
                  (enriched['email_opens_30d'] > 3)].head(2)

# Case type 5: Discount-Dependent with decent frequency
case5 = enriched[(enriched['segment_name']=='Discount-Dependent')].head(1)

manual_cases = pd.concat([case1, case2, case3, case4, case5]).head(10)

review_cols = ['customer_id','segment_name','recency_days','frequency','monetary',
               'ticket_count','negative_tickets','avg_sentiment','sessions_30d',
               'return_rate','avg_discount','churn_next_60d']

print(f'Manual Review: {len(manual_cases)} customers selected')
display(manual_cases[review_cols])

In [ ]:
# ── Generate reasoning for each case ─────────────────────────────────────────
print('\n=== MANUAL REVIEW CASE REASONING ===')
for i, (_, row) in enumerate(manual_cases.iterrows(), 1):
    cid = row['customer_id']
    seg = row['segment_name']
    rec = row['recency_days']
    freq = row['frequency']
    mon = row['monetary']
    tix = row['ticket_count']
    neg = row['negative_tickets']
    sess = row['sessions_30d']
    ret_rate = row['return_rate']
    churned = row['churn_next_60d']
    
    print(f'\n--- Case {i}: {cid} (Segment: {seg}) ---')
    print(f'  Recency: {rec:.0f}d | Freq: {freq} | Spend: ₹{mon:,.0f} | Tickets: {tix} (neg: {neg}) | Sessions: {sess} | Return rate: {ret_rate:.0%}')
    print(f'  Actual churn: {"YES" if churned else "NO"}')
    
    if seg == 'At-Risk' and sess > 3:
        print(f'  DILEMMA: Classified At-Risk by recency, but still browsing the site ({sess} sessions).')
        print(f'  RECOMMENDATION: This customer is window-shopping but not buying. Send a personalised')
        print(f'  product recommendation based on their browsing. Do NOT send a blanket discount.')
    elif seg == 'Champions' and neg >= 1:
        print(f'  DILEMMA: Top customer with {neg} negative ticket(s). High churn risk despite current engagement.')
        print(f'  RECOMMENDATION: Escalate support case immediately. Offer proactive apology + account credit.')
        print(f'  Losing a Champion costs far more than the credit.')
    elif seg == 'New Customers' and mon > enriched['monetary'].quantile(0.8):
        print(f'  DILEMMA: New customer but already spent ₹{mon:,.0f}. Could become a Champion or one-time gifter.')
        print(f'  RECOMMENDATION: Send personalised onboarding + ask for category preferences.')
        print(f'  If second order comes within 30 days, fast-track to loyalty programme.')
    elif seg == 'Dormant' and row['email_opens_30d'] > 3:
        print(f'  DILEMMA: No purchases but still opens {row["email_opens_30d"]} marketing emails. Interested but not converting.')
        print(f'  RECOMMENDATION: This is a re-activation opportunity. Send a "We miss you" offer with a')
        print(f'  specific product from their previous category. Time-bound (72 hours).')
    elif seg == 'Discount-Dependent':
        print(f'  DILEMMA: {freq} orders but avg discount is {row["avg_discount"]:.0%}. Only buys on sale.')
        print(f'  RECOMMENDATION: Do NOT send another discount. Instead, offer loyalty points or cashback')
        print(f'  to shift the value perception away from discounting.')
    else:
        print(f'  DILEMMA: Mixed signals — segment placement may not capture full picture.')
        print(f'  RECOMMENDATION: Flag for CRM team manual review based on recent browsing history.')

## 8. Export segments.csv

In [ ]:
# ── Export final segments.csv ──────────────────────────────────────────────────
export_cols = ['customer_id', 'segment_name', 'recency_days', 'frequency', 'monetary',
               'R_score', 'F_score', 'M_score', 'RFM_score',
               'ticket_count', 'avg_sentiment', 'sessions_30d',
               'return_rate', 'avg_discount']

segments_csv = enriched[export_cols].copy()
segments_csv.to_csv('segments.csv', index=False)

print(f'segments.csv exported: {segments_csv.shape[0]:,} rows × {segments_csv.shape[1]} columns')
print(f'\nSegment counts in export:')
print(segments_csv['segment_name'].value_counts())
print(f'\nFirst 5 rows:')
display(segments_csv.head())